# 01 — Baseline ảnh: CNN & Transformer (MILK10k)

**Stage 1.** 4 backbone bản base × 3 nhánh ảnh, cùng loss mặc định (CE + trọng số √(1/tần suất)), val trên **fold 0**.

| Backbone | Họ | Tham số | Ảnh |
|---|---|---|---|
| ResNet-152 | CNN | 60M | 320px |
| ConvNeXt-Base | CNN hiện đại | 89M | 224px |
| ViT-Base/16 | Transformer | 86M | 224px |
| Swin-Base | Transformer | 88M | 224px |

Mỗi backbone chạy lần lượt: **clinical** → **dermoscopy** → **clinical + dermoscopy** (12 run).
Mỗi run tạo `outputs/<backbone>__ce_sqrtinv__<view>/` gồm `train.log`, `history.csv`, `metrics.json`, `submission.csv`.
Tên run: `<backbone>__<tag imbalance>__<nhánh ảnh>`.

### Cách chạy trên Kaggle
1. **Settings → Accelerator: GPU T4 x2**, **Internet: On**.
2. Ô *Cấu hình* đã có sẵn link repo và link dữ liệu; chỉ cần sửa `RESTORE` khi chạy tiếp.
3. **Save Version → Save & Run All (Commit)**: notebook chạy nền tối đa 12 giờ, không cần mở trình duyệt.
   Toàn bộ `outputs/` (log, bảng, file nộp) được lưu vào output của version.
4. **Chưa chạy hết lưới** (hết 12 giờ): Add Input → output của version vừa chạy, điền vào `RESTORE`,
   rồi Save Version lần nữa. Các run đã xong được bỏ qua, run dở dang được chạy lại.

Hai GPU được dùng song song: mỗi GPU chạy một run độc lập (`scripts/run_grid.py`). Checkpoint `.pt` bị xoá
sau khi sinh file nộp để không vượt giới hạn 20GB output; `test_probs.csv` được giữ lại để ensemble.

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL = "https://github.com/trong5nhan6/fussion.git"
BRANCH = "main"
DRIVE_URL = "https://drive.google.com/file/d/1lM1nB_4pq3Vg-GwlaTMQjtus_bpbMhGX/view?usp=drive_link"  # zip dữ liệu MILK10k
DATA_SOURCE = None    # hoặc đường dẫn Kaggle Dataset, vd "/kaggle/input/milk10k" (khi đó không tải từ Drive)

# Chạy tiếp / dùng lại kết quả: gắn output của notebook trước làm Input (Add Input -> Your Work -> Notebooks),
# rồi liệt kê thư mục outputs của nó, vd "/kaggle/input/<ten-notebook>/fussion/outputs"
RESTORE = []
TIME_BUDGET_H = 11.0   # dừng nhận run mới để kịp lưu trước giới hạn 12 giờ của Kaggle

In [ ]:
# ===================== THAM SỐ TRAIN =====================
# Áp dụng cho MỌI run của lưới, ưu tiên cao nhất (ghi đè configs/default.yaml, configs/baselines/*, configs/env/kaggle_t4.yaml).
# None = giữ giá trị trong config. Một số tham số khác nhau theo backbone, chỉ đặt khi muốn áp dụng đồng loạt:
#   img_size: ResNet-152 320, ConvNeXt/ViT/Swin 224 (Swin-B window 7 bắt buộc 224)
#   batch_size x accum_steps: 16 x 2 (ResNet-152: 8 x 4), batch hiệu dụng 32
#   lr: ResNet-152 3e-4, ConvNeXt-B 1e-4, ViT-B / Swin-B 5e-5
# Xem bảng "Cấu hình thực tế" ở bước Danh sách run để kiểm tra trước khi train.
PARAMS = {
    # --- Huấn luyện ---
    "train.epochs": 20,                # số epoch tối đa
    "train.patience": 5,               # dừng sớm sau N epoch Dice val không tăng
    "train.warmup_epochs": 1,          # warmup tuyến tính rồi cosine
    "train.lr": None,                  # lr của head; backbone = lr * backbone_lr_mult
    "train.backbone_lr_mult": None,
    "train.weight_decay": None,
    "train.batch_size": None,
    "train.accum_steps": None,
    "train.amp": True,                 # mixed precision fp16
    "train.monitor": "dice",           # metric chọn checkpoint: dice | auc | ap | balanced_acc
    "train.folds": [0],                # [0, 1, 2, 3, 4] khi đã chốt method và chạy đủ 5 fold
    "seed": 42,
    # --- Dữ liệu & mô hình ---
    "data.img_size": None,
    "model.pretrained": True,
    "model.dropout": None,
    # --- Đánh giá & file nộp ---
    "predict.tta": True,               # trung bình ảnh gốc + lật ngang + lật dọc
    "predict.postprocess": "top1",     # top1 (khuyến nghị) | softmax
}

# Đổi tham số so với lần chạy trước -> đặt hậu tố, vd "ep30". Run trùng tên đã xong sẽ bị BỎ QUA,
# nên không có hậu tố thì kết quả cũ được giữ nguyên thay vì train lại.
RUN_SUFFIX = ""

## 1. Cài đặt

In [ ]:
import os, glob
from IPython.display import Markdown, display

REPO_DIR = "/kaggle/working/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
DATA_DIR = "/tmp/MILK10k"               # ngoài /kaggle/working -> dữ liệu không bị lưu vào output notebook
os.environ["MILK10K_ROOT"] = DATA_DIR   # code tự đọc biến này thay cho data.root trong config

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q gdown
!git log -1 --format="Commit: %h %s"

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| số GPU:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 1, "Chưa bật GPU: Settings -> Accelerator -> GPU T4 x2"

## 2. Dữ liệu

In [ ]:
src = f"--source {DATA_SOURCE}" if DATA_SOURCE else f'--drive-url "{DRIVE_URL}"'
!python scripts/setup_data.py {src} --dest {DATA_DIR}

In [ ]:
# Kiểm tra nhanh toàn bộ pipeline (không cần GPU, ~1 phút)
!python -m pytest tests -q -p no:cacheprovider

## 3. Danh sách run và cấu hình thực tế

In [ ]:
restore_arg = ("--restore " + " ".join(RESTORE)) if RESTORE else ""
for r in RESTORE:
    print(r, "->", "OK" if os.path.isdir(r) else "KHÔNG TỒN TẠI (kiểm tra lại đường dẫn Input)")

In [ ]:
import shlex, yaml

def to_set_args(params: dict) -> str:
    # Chuyển PARAMS thành chuỗi --set key=value cho scripts (bỏ qua giá trị None)
    items = []
    for k, v in params.items():
        if v is None:
            continue
        val = yaml.safe_dump(v, default_flow_style=True).strip().removesuffix("...").strip()
        items.append(shlex.quote(f"{k}={val}"))
    return ("--set " + " ".join(items)) if items else ""

set_arg = to_set_args(PARAMS)
suffix_arg = f"--suffix {RUN_SUFFIX}" if RUN_SUFFIX else ""
print("Override:", set_arg or "(không có)")
print("Hậu tố tên run:", RUN_SUFFIX or "(không có)")

In [ ]:
!python scripts/run_grid.py experiments/stage1_baselines.yaml --env configs/env/kaggle_t4.yaml {set_arg} {suffix_arg} {restore_arg} --dry-run

## 4. Train
Log từng epoch của cả 2 GPU được in bên dưới; log đầy đủ của từng run: `outputs/<run>/train.log`.
Nếu một run lỗi hết bộ nhớ (OOM), giảm batch cho backbone đó trong `experiments/stage1_baselines.yaml`
(vd `set: [train.batch_size=8, train.accum_steps=4]`) rồi chạy lại.

In [ ]:
!python scripts/run_grid.py experiments/stage1_baselines.yaml --env configs/env/kaggle_t4.yaml {set_arg} {suffix_arg} {restore_arg} --gpus auto --time-budget-h {TIME_BUDGET_H}

## 5. Kết quả

In [ ]:
!python scripts/summarize.py --dirs outputs > /dev/null
display(Markdown(open("outputs/summary/summary.md", encoding="utf-8").read()))
print("Chọn 1–2 backbone có Dice cao nhất (cột clin+derm) cho notebook 02_imbalance.")